**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 14: 파인튜닝 개념 정리 - SFT/RL, CPT/IT, LoRA/QLoRA

**Part 3: 파인튜닝 기초** | GPU: No (이론 중심)

---

### 📋 학습 목표
- 🎯 LLM 학습의 전체 파이프라인(Pretraining → SFT → RLHF)을 이해한다
- 🎯 Continuous Pretraining과 Instruction Tuning의 차이를 파악한다
- 🎯 LoRA/QLoRA의 원리와 수학적 배경을 학습한다
- 🎯 RTX 4060 환경에서 학습 가능한 모델 범위를 파악한다

### ⏱️ 예상 소요 시간: 90분

In [ ]:
# 이 노트북은 이론 중심이므로 GPU가 필요하지 않습니다
import numpy as np
import json

print("✅ Session 14: 파인튜닝 개념 정리")
print("📌 이 노트북은 이론/개념 중심으로, GPU가 필요하지 않습니다.")
print("📌 간단한 계산과 시각화만 포함되어 있습니다.")

---
## 🎯 1. 파인튜닝이란?

### 정의
**파인튜닝(Fine-tuning)**이란 사전학습된 모델(Pretrained Model)을 **특정 목적에 맞게 추가 학습**시키는 과정입니다.

### 왜 파인튜닝이 필요한가?

| 사전학습 모델 | 파인튜닝 후 |
|-------------|------------|
| 범용적인 언어 이해 | 특정 도메인 전문 지식 |
| 일반적인 답변 | 원하는 형식의 응답 |
| 넓지만 얕은 지식 | 좁지만 깊은 전문성 |
| 영어 중심 | 한국어 특화 |

### 비유: 대학 졸업생 vs 전문가

```
🎓 사전학습 = 대학 교육 (범용적 지식 습득)
🔧 파인튜닝 = 직무 교육 (특정 업무 능력 향상)
```

- 📌 대학을 졸업한 사람(사전학습된 모델)은 기본기가 탄탄
- 📌 하지만 회사에서 바로 일하려면 직무 교육(파인튜닝)이 필요
- 📌 직무 교육은 대학 교육보다 훨씬 짧고 적은 비용으로 가능!

---
## 1️⃣ 2. 학습 단계: Pretraining → SFT → RLHF

LLM은 일반적으로 **3단계**에 걸쳐 학습됩니다.

### Stage 1: Pretraining (사전학습)

```
📚 인터넷 텍스트 (수조 토큰)
    ↓
🔄 Next Token Prediction (자기지도학습)
    ↓
🧠 Base Model (언어 이해력 습득)
```

- 📌 **데이터**: 인터넷 크롤링, 책, 논문, 코드 등 (수조 토큰)
- 📌 **방법**: Next Token Prediction (다음 토큰 예측)
- 📌 **비용**: 수십~수백만 달러, 수천 GPU-hours
- 📌 **결과**: Base Model (문장 완성은 가능하지만 대화 불가)

### Stage 2: SFT (Supervised Fine-Tuning)

```
👤 [질문-답변 쌍 데이터]
    ↓
🔄 지도학습 (정답을 보여주며 학습)
    ↓
🤖 Instruction-following Model (지시를 따르는 모델)
```

- 📌 **데이터**: (질문, 답변) 쌍 수만~수십만개
- 📌 **방법**: 정답 응답을 학습 (지도학습)
- 📌 **비용**: 사전학습의 1% 미만
- 📌 **결과**: 사용자 지시를 따르는 모델

### Stage 3: RLHF (Reinforcement Learning from Human Feedback)

```
👥 [인간 선호도 데이터]
    ↓
🔄 강화학습 (인간 피드백으로 개선)
    ↓
🌟 Aligned Model (안전하고 유용한 모델)
```

- 📌 **데이터**: 응답 A vs B 중 인간이 선호하는 것 선택
- 📌 **방법**: PPO, DPO 등 강화학습 알고리즘
- 📌 **목적**: 유해한 응답 억제, 유용성 향상, 인간 가치와 정렬

In [ ]:
# 각 단계별 비용/데이터 비교
print("📊 LLM 학습 3단계 비교")
print("=" * 70)

stages = [
    {
        "stage": "1. Pretraining",
        "data_size": "1~15조 토큰",
        "data_type": "인터넷 텍스트 (비지도)",
        "cost": "$1M ~ $100M+",
        "gpu_hours": "수십만 GPU-hours",
        "result": "Base Model"
    },
    {
        "stage": "2. SFT",
        "data_size": "1만 ~ 100만 샘플",
        "data_type": "(질문, 답변) 쌍",
        "cost": "$100 ~ $10K",
        "gpu_hours": "수십 GPU-hours",
        "result": "Chat Model"
    },
    {
        "stage": "3. RLHF",
        "data_size": "1만 ~ 10만 쌍",
        "data_type": "선호도 비교 데이터",
        "cost": "$1K ~ $50K",
        "gpu_hours": "수백 GPU-hours",
        "result": "Aligned Model"
    }
]

for s in stages:
    print(f"\n{'='*60}")
    print(f"📌 {s['stage']}")
    print(f"{'='*60}")
    print(f"  데이터 규모: {s['data_size']}")
    print(f"  데이터 유형: {s['data_type']}")
    print(f"  비용:       {s['cost']}")
    print(f"  GPU 시간:   {s['gpu_hours']}")
    print(f"  결과물:     {s['result']}")

print("\n" + "=" * 70)
print("💡 핵심: 우리가 실습할 파인튜닝은 Stage 2 (SFT)에 해당합니다!")
print("   → 비교적 적은 데이터와 비용으로 큰 효과를 낼 수 있습니다.")

---
## 2️⃣ 3. Continuous Pretraining (CPT) 개념

### CPT란?
사전학습된 모델에 **추가적인 도메인 텍스트를 계속 학습**시키는 것

```
🧠 Base Model (일반 지식)
    ↓ + 의학 논문 100만개
🏥 Medical Model (의학 지식 강화)
```

### 언제 CPT를 사용하나?

| 상황 | CPT 필요성 | 예시 |
|------|----------|------|
| 도메인 전문 용어가 많은 경우 | ⭐⭐⭐ | 의학, 법률, 금융 |
| 한국어 성능 강화 | ⭐⭐⭐ | 영어 기반 모델 → 한국어 |
| 최신 정보 반영 | ⭐⭐ | 학습 데이터 이후의 뉴스/정보 |
| 일반적인 작업 | ⭐ | SFT만으로 충분 |

### CPT vs SFT 비교

| 항목 | CPT | SFT |
|------|-----|-----|
| **목적** | 도메인 지식 습득 | 지시 수행 능력 |
| **데이터 형식** | 일반 텍스트 (비구조화) | (질문, 답변) 쌍 |
| **학습 방법** | Next Token Prediction | Next Token Prediction |
| **데이터 양** | 많을수록 좋음 (수억 토큰) | 수만~수십만 개 |
| **학습률** | 매우 낮음 (1e-5) | 약간 높음 (1e-4~3e-4) |

In [ ]:
# CPT 데이터 예시
print("📌 CPT 데이터 예시: 도메인 특화 텍스트")
print("=" * 60)

cpt_examples = {
    "의학 CPT": [
        "당뇨병은 인슐린 분비 또는 작용의 결함으로 인해 발생하는 대사 질환이다.",
        "제2형 당뇨병의 일차 치료제로 메트포르민이 권장된다.",
        "HbA1c 수치가 6.5% 이상이면 당뇨병으로 진단한다."
    ],
    "법률 CPT": [
        "민법 제750조에 따르면 고의 또는 과실로 인한 위법행위로 타인에게 손해를 가한 자는 그 손해를 배상할 책임이 있다.",
        "채권자대위권은 채무자가 그의 권리를 행사하지 아니하는 때에 채권자가 자기의 채권을 보전하기 위하여 채무자의 권리를 행사할 수 있는 제도이다."
    ],
    "한국어 CPT": [
        "서울의 인구는 약 950만 명으로 대한민국 전체 인구의 약 18%를 차지한다.",
        "한글은 세종대왕이 1443년에 창제하고 1446년에 반포한 문자 체계이다.",
        "김치는 배추, 무 등의 채소를 양념에 버무려 발효시킨 한국의 전통 음식이다."
    ]
}

for domain, examples in cpt_examples.items():
    print(f"\n🏷️ {domain}:")
    for i, ex in enumerate(examples, 1):
        print(f"   {i}. {ex[:80]}{'...' if len(ex) > 80 else ''}")

print("\n💡 CPT 데이터 특징:")
print("  - 질문-답변 형식이 아닌 일반 텍스트")
print("  - 도메인 전문 용어와 지식이 풍부")
print("  - 양이 많을수록 효과적 (수억 토큰 권장)")

---
## 3️⃣ 4. Instruction Tuning (IT) 개념

### Instruction Tuning이란?
모델이 **사용자의 지시(Instruction)를 이해하고 따르도록** 학습시키는 것

```
👤 사용자: "다음 텍스트를 요약해줘: ..."
🤖 모델: "요약 결과: ..."
```

### SFT vs Instruction Tuning

사실 **SFT(Supervised Fine-Tuning)**와 **Instruction Tuning**은 동일한 개념입니다!
- 📌 SFT = 학습 방법론 (지도학습 기반 파인튜닝)
- 📌 Instruction Tuning = SFT의 목적 (지시 수행 능력 부여)

### Instruction Tuning 데이터 형식

```json
{
    "instruction": "다음 문장을 영어로 번역해주세요.",
    "input": "오늘 날씨가 좋습니다.",
    "output": "The weather is nice today."
}
```

### 다양한 Instruction 유형

| 유형 | 예시 |
|------|------|
| 요약 | "다음 기사를 3줄로 요약해줘" |
| 번역 | "한국어를 영어로 번역해줘" |
| 분류 | "이 리뷰의 감성을 분석해줘" |
| 생성 | "주어진 키워드로 이메일을 작성해줘" |
| 질의응답 | "제2차 세계대전은 언제 끝났어?" |
| 코딩 | "Python으로 정렬 알고리즘을 구현해줘" |

In [ ]:
# Instruction Tuning 데이터 예시
print("📌 Instruction Tuning 데이터 예시")
print("=" * 60)

instruction_examples = [
    {
        "instruction": "다음 텍스트의 감성을 분석하세요.",
        "input": "이 영화는 정말 재미있었어요! 배우들의 연기도 훌륭했습니다.",
        "output": "이 텍스트의 감성은 **긍정적**입니다. '재미있었어요', '훌륭했습니다' 등의 긍정적 표현이 사용되었습니다."
    },
    {
        "instruction": "주어진 내용을 3줄로 요약하세요.",
        "input": "인공지능(AI)은 1950년대부터 연구되어 왔으며, 최근 딥러닝의 발전으로 급격한 성능 향상을 이루었습니다. 특히 GPT, BERT 같은 트랜스포머 기반 모델들이 자연어처리 분야를 혁신했습니다.",
        "output": "1. AI는 1950년대부터 연구된 분야입니다.\n2. 딥러닝의 발전으로 최근 급격한 성능 향상이 있었습니다.\n3. GPT, BERT 등 트랜스포머 모델이 NLP를 혁신했습니다."
    },
    {
        "instruction": "주어진 단어의 의미를 설명하세요.",
        "input": "파인튜닝",
        "output": "파인튜닝(Fine-tuning)은 사전학습된 AI 모델을 특정 작업이나 도메인에 맞게 추가 학습시키는 기법입니다. 적은 데이터와 비용으로 모델의 성능을 특정 목적에 맞게 최적화할 수 있습니다."
    }
]

for i, ex in enumerate(instruction_examples, 1):
    print(f"\n{'─'*60}")
    print(f"📋 예시 {i}:")
    print(f"  🔹 Instruction: {ex['instruction']}")
    print(f"  🔹 Input: {ex['input'][:80]}{'...' if len(ex['input']) > 80 else ''}")
    print(f"  🔹 Output: {ex['output'][:80]}{'...' if len(ex['output']) > 80 else ''}")

print(f"\n\n💡 Instruction Tuning의 핵심:")
print(f"  - 다양한 유형의 지시를 학습시켜야 일반화 성능이 높아짐")
print(f"  - 데이터 품질 > 데이터 양 (고품질 1만개 > 저품질 100만개)")
print(f"  - LIMA 논문: 단 1,000개의 고품질 데이터만으로도 좋은 성능!")

---
## 4️⃣ 5. Full Fine-tuning vs Parameter-Efficient Fine-tuning (PEFT)

### Full Fine-tuning (FFT)
모델의 **모든 파라미터**를 업데이트하는 방식

```
전체 파라미터 ✅ 업데이트
장점: 최고의 성능 가능
단점: 매우 많은 VRAM 필요
```

### Parameter-Efficient Fine-tuning (PEFT)
모델의 **일부 파라미터만** 업데이트하는 방식

```
기존 파라미터 ❄️ 동결 (Frozen)
추가 파라미터 ✅ 학습 (매우 적은 수)
장점: 적은 VRAM으로 학습 가능
```

### 비교

| 항목 | FFT | PEFT (LoRA) |
|------|-----|-------------|
| 학습 파라미터 | 100% | 0.1~5% |
| VRAM 사용량 | 매우 높음 | 낮음 |
| 성능 | 최고 | FFT의 90~99% |
| 학습 속도 | 느림 | 빠름 |
| RTX 4060 가능? | 1.5B까지만 | 7B까지 (QLoRA) |

In [ ]:
# FFT vs PEFT VRAM 사용량 비교
print("📊 FFT vs LoRA VRAM 사용량 비교")
print("=" * 60)

# VRAM 계산 함수
def calculate_vram(params_b, method="fft", precision="fp16"):
    """
    VRAM 추정 계산
    - 모델 가중치: params × bytes_per_param
    - 옵티마이저 상태: Adam은 파라미터의 2배
    - 그래디언트: 학습 파라미터와 동일 크기
    - 활성화 메모리: 배치/시퀀스에 비례
    """
    bytes_per_param = 2 if precision == "fp16" else 4  # fp16=2B, fp32=4B
    
    model_size_gb = params_b * 1e9 * bytes_per_param / 1e9
    
    if method == "fft":
        # FFT: 모델 + 그래디언트 + 옵티마이저(2x) ≈ 모델의 4배
        optimizer_gb = params_b * 1e9 * 4 * 2 / 1e9  # Adam: fp32로 2개 상태
        gradient_gb = model_size_gb
        activation_gb = model_size_gb * 0.3  # 근사치
        total_gb = model_size_gb + optimizer_gb + gradient_gb + activation_gb
    elif method == "lora":
        # LoRA: 모델 + 작은 LoRA 파라미터의 그래디언트/옵티마이저
        lora_ratio = 0.02  # ~2% 파라미터
        lora_params = params_b * lora_ratio
        optimizer_gb = lora_params * 1e9 * 4 * 2 / 1e9
        gradient_gb = lora_params * 1e9 * bytes_per_param / 1e9
        activation_gb = model_size_gb * 0.2
        total_gb = model_size_gb + optimizer_gb + gradient_gb + activation_gb
    elif method == "qlora":
        # QLoRA: 4bit 모델 + LoRA
        model_size_gb = params_b * 1e9 * 0.5 / 1e9  # 4bit = 0.5 bytes
        lora_ratio = 0.02
        lora_params = params_b * lora_ratio
        optimizer_gb = lora_params * 1e9 * 4 * 2 / 1e9
        gradient_gb = lora_params * 1e9 * 2 / 1e9
        activation_gb = model_size_gb * 0.3
        total_gb = model_size_gb + optimizer_gb + gradient_gb + activation_gb
    
    return total_gb

models = [1.5, 3, 7, 13, 70]
methods = ["fft", "lora", "qlora"]
method_names = ["Full Fine-tuning (fp16)", "LoRA (fp16)", "QLoRA (4bit)"]

print(f"{'모델 크기':>10s}", end="")
for name in method_names:
    print(f" | {name:>25s}", end="")
print()
print("-" * 95)

for model_size in models:
    print(f"{model_size:>8.1f}B", end="")
    for method in methods:
        vram = calculate_vram(model_size, method)
        rtx4060 = "✅" if vram <= 8 else "❌"
        print(f" | {vram:>18.1f}GB {rtx4060:>4s}", end="")
    print()

print("\n" + "=" * 95)
print("📌 RTX 4060 (8GB VRAM) 기준:")
print("  - FFT: 1.5B까지만 가능")
print("  - LoRA: 3B까지 가능")
print("  - QLoRA: 7B까지 빠듯하게 가능")
print("\n⚠️ 위 수치는 추정값이며, 실제로는 시퀀스 길이/배치 사이즈에 따라 달라집니다.")

---
## 5️⃣ 6. LoRA 원리 상세 설명

### LoRA (Low-Rank Adaptation) 란?

2021년 Microsoft에서 발표한 논문 "LoRA: Low-Rank Adaptation of Large Language Models"

### 핵심 아이디어

기존 가중치 행렬 $W$를 직접 수정하는 대신, **저차원(Low-Rank) 행렬 2개의 곱**으로 변화량을 표현

$$W' = W + \Delta W = W + BA$$

여기서:
- 📌 $W \in \mathbb{R}^{d \times k}$ : 원본 가중치 (동결, 학습 안 함)
- 📌 $B \in \mathbb{R}^{d \times r}$ : LoRA 행렬 B (학습)
- 📌 $A \in \mathbb{R}^{r \times k}$ : LoRA 행렬 A (학습)
- 📌 $r$ : 랭크 (rank) - 보통 8, 16, 32, 64

### 파라미터 수 비교

```
원본 가중치 W: d × k 파라미터
LoRA (B, A): d × r + r × k = r × (d + k) 파라미터

예시: d=4096, k=4096, r=16
  원본: 4096 × 4096 = 16,777,216 (약 1677만)
  LoRA: 16 × (4096 + 4096) = 131,072 (약 13만)
  비율: 0.78% ← 원본의 1%도 안 됨!
```

### LoRA 구조 다이어그램

```
                    ┌─────────────┐
                    │  x (입력)    │
                    └──────┬──────┘
                           │
              ┌────────────┼────────────┐
              │            │            │
              ▼            │            ▼
    ┌──────────────┐      │   ┌──────────────┐
    │  W (동결 ❄️)  │      │   │  A (d→r) 🔥  │
    │  d × k       │      │   │  학습 가능     │
    └──────┬───────┘      │   └──────┬───────┘
           │              │          │
           │              │          ▼
           │              │   ┌──────────────┐
           │              │   │  B (r→k) 🔥  │
           │              │   │  학습 가능     │
           │              │   └──────┬───────┘
           │              │          │
           └──────┬───────┘──────────┘
                  │ (합산)
                  ▼
           ┌──────────────┐
           │  h = Wx + BAx│
           └──────────────┘
```

### 주요 하이퍼파라미터

| 파라미터 | 역할 | 권장값 |
|---------|------|-------|
| **r (rank)** | LoRA 행렬의 차원 | 8~64 (보통 16) |
| **lora_alpha** | 스케일링 계수 | r과 같거나 2배 (16~32) |
| **lora_dropout** | 드롭아웃 비율 | 0.05~0.1 |
| **target_modules** | LoRA 적용 대상 | q_proj, v_proj, k_proj, o_proj |

In [ ]:
# LoRA 파라미터 계산기
print("📊 LoRA 파라미터 계산기")
print("=" * 60)

def lora_params_calculator(d, k, r, n_layers, n_modules=4):
    """
    LoRA 파라미터 수 계산
    d: hidden dimension
    k: output dimension (보통 d와 같음)
    r: LoRA rank
    n_layers: 트랜스포머 레이어 수
    n_modules: LoRA 적용 모듈 수 (q,k,v,o_proj)
    """
    original_per_module = d * k
    lora_per_module = r * (d + k)
    
    total_original = original_per_module * n_layers * n_modules
    total_lora = lora_per_module * n_layers * n_modules
    
    return total_original, total_lora

# 실제 모델 스펙
models_spec = {
    "Qwen2.5-1.5B": {"d": 1536, "k": 1536, "layers": 28, "total_params": 1.5e9},
    "Qwen2.5-3B": {"d": 2048, "k": 2048, "layers": 36, "total_params": 3e9},
    "Qwen2.5-7B": {"d": 3584, "k": 3584, "layers": 28, "total_params": 7e9},
    "Llama-3-8B": {"d": 4096, "k": 4096, "layers": 32, "total_params": 8e9},
}

ranks = [8, 16, 32, 64]

print(f"\n{'모델':>20s} | {'r':>4s} | {'원본 파라미터':>15s} | {'LoRA 파라미터':>15s} | {'비율':>8s}")
print("-" * 80)

for model_name, spec in models_spec.items():
    for r in ranks:
        orig, lora = lora_params_calculator(
            spec['d'], spec['k'], r, spec['layers']
        )
        ratio = lora / spec['total_params'] * 100
        print(f"{model_name:>20s} | {r:>4d} | {spec['total_params']/1e9:>12.1f}B | {lora/1e6:>12.2f}M | {ratio:>6.2f}%")
    print("-" * 80)

print("\n💡 핵심:")
print("  - rank=16이면 전체 파라미터의 약 0.5~1%만 학습!")
print("  - 학습 파라미터가 적으므로 VRAM, 학습 시간, 저장 공간 절약")
print("  - rank를 높이면 표현력 증가하지만 과적합 위험도 증가")

In [ ]:
# LoRA alpha와 스케일링 이해
print("📊 LoRA alpha와 스케일링 이해")
print("=" * 60)

print("""
🔑 LoRA의 실제 적용 공식:

    h = Wx + (alpha / r) × BAx
    
여기서 (alpha / r)이 스케일링 팩터입니다.

""")

# 스케일링 팩터 비교
print(f"{'r':>6s} | {'alpha':>8s} | {'스케일링 (alpha/r)':>20s} | {'설명':>20s}")
print("-" * 65)

configs = [
    (8, 8, "표준 (변화량 1배)"),
    (8, 16, "변화량 2배"),
    (16, 16, "표준 (변화량 1배)"),
    (16, 32, "변화량 2배"),
    (32, 32, "표준 (변화량 1배)"),
    (64, 16, "변화량 약화 (0.25배)"),
]

for r, alpha, desc in configs:
    scaling = alpha / r
    print(f"{r:>6d} | {alpha:>8d} | {scaling:>20.2f} | {desc:>20s}")

print("\n💡 실무 팁:")
print("  - 일반적으로 alpha = r 또는 alpha = 2*r로 설정")
print("  - alpha가 크면 → LoRA의 영향력 증가 (더 많이 변화)")
print("  - alpha가 작으면 → LoRA의 영향력 감소 (원본에 가까움)")
print("  - 실습에서는 r=16, alpha=32를 기본으로 사용합니다")

---
## 6️⃣ 7. QLoRA 원리 (4bit NF4 양자화 + LoRA)

### QLoRA란?

2023년 워싱턴 대학에서 발표한 "QLoRA: Efficient Finetuning of Quantized LLMs"

**핵심**: 모델을 4bit로 양자화하여 로드한 후, LoRA로 학습

### QLoRA의 3가지 핵심 기술

#### 1. NF4 (4-bit NormalFloat) 양자화
- 📌 정규분포에 최적화된 4비트 데이터 타입
- 📌 FP16(16bit) → NF4(4bit)로 압축 = **4배 메모리 절약**
- 📌 성능 저하 최소화 (일반 INT4보다 우수)

#### 2. Double Quantization
- 📌 양자화 상수(quantization constants)까지 다시 양자화
- 📌 추가 메모리 절약 (파라미터당 약 0.37bit 절약)

#### 3. Paged Optimizers
- 📌 GPU 메모리 부족 시 CPU로 자동 스왑
- 📌 OOM(Out-of-Memory) 에러 방지

### 메모리 비교 (7B 모델 기준)

| 방법 | 모델 메모리 | 학습 가능 | 총 VRAM |
|------|-----------|---------|--------|
| FFT (fp16) | 14GB | ❌ (VRAM 부족) | ~56GB |
| LoRA (fp16) | 14GB | ❌ (모델 로드 불가) | ~18GB |
| **QLoRA (4bit)** | **3.5GB** | ✅ | **~6-8GB** |

In [ ]:
# 양자화 비트별 메모리 비교
print("📊 양자화 비트별 메모리 사용량 비교")
print("=" * 60)

def model_memory_gb(params_b, bits):
    """모델 로드에 필요한 메모리 계산"""
    bytes_per_param = bits / 8
    return params_b * bytes_per_param

model_sizes = [1.5, 3, 7, 13, 70]
bit_configs = [
    (32, "FP32"),
    (16, "FP16/BF16"),
    (8, "INT8"),
    (4, "NF4 (QLoRA)"),
]

print(f"{'모델':>8s}", end="")
for _, name in bit_configs:
    print(f" | {name:>15s}", end="")
print()
print("-" * 80)

for size in model_sizes:
    print(f"{size:>6.1f}B", end="")
    for bits, _ in bit_configs:
        mem = model_memory_gb(size, bits)
        fit = "✅" if mem <= 8 else "❌"
        print(f" | {mem:>10.1f}GB {fit:>2s}", end="")
    print()

print("\n📌 RTX 4060 (8GB VRAM) 기준 ✅ 표시")
print("\n💡 QLoRA의 핵심 장점:")
print("  - 7B 모델을 3.5GB로 로드 가능 → RTX 4060에서 학습 가능!")
print("  - NF4 양자화는 정규분포에 최적화되어 성능 저하 최소")
print("  - bitsandbytes 라이브러리로 간편하게 적용")

In [ ]:
# QLoRA 코드 예시 (실행 불가 - 참고용)
print("📝 QLoRA 설정 코드 예시 (참고용)")
print("=" * 60)

qlora_code = '''
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

# 1. 4bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # 4bit로 로드
    bnb_4bit_quant_type="nf4",      # NormalFloat 4bit
    bnb_4bit_compute_dtype=torch.float16,  # 연산은 fp16
    bnb_4bit_use_double_quant=True,  # Double Quantization
)

# 2. 모델 로드 (4bit)
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)

# 3. LoRA 설정
lora_config = LoraConfig(
    r=16,                    # LoRA rank
    lora_alpha=32,           # 스케일링 (alpha/r = 2)
    lora_dropout=0.05,       # 드롭아웃
    target_modules=[         # LoRA 적용 대상
        "q_proj", "k_proj", 
        "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

# 4. LoRA 적용
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 출력: trainable params: 25,165,824 || all params: 7,640,641,536 || 0.33%
'''

print(qlora_code)

print("💡 위 코드는 Session 17에서 실제로 실행합니다!")
print("   지금은 각 설정값의 의미를 이해하는 것이 중요합니다.")

---
## 7️⃣ 8. RTX 4060 (8GB)에서의 모델별 학습 가능 범위

### 실습 환경 제약 정리

| 모델 크기 | FFT | LoRA (fp16) | QLoRA (4bit) | 추론만 |
|----------|-----|-------------|-------------|-------|
| **0.5B** | ✅ | ✅ | ✅ | ✅ |
| **1.5B** | ✅ (빠듯) | ✅ | ✅ | ✅ |
| **3B** | ❌ | ✅ (빠듯) | ✅ | ✅ |
| **7B** | ❌ | ❌ | ✅ (빠듯) | ✅ (4bit) |
| **13B** | ❌ | ❌ | ❌ | ❌ |

### 권장 실습 모델

| 용도 | 권장 모델 | 방법 |
|------|---------|------|
| FFT 실습 | Qwen2.5-1.5B-Instruct | FFT (fp16) |
| LoRA 실습 | Qwen2.5-1.5B-Instruct | LoRA (fp16) |
| QLoRA 실습 | Qwen2.5-7B-Instruct | QLoRA (4bit) |
| LoRA vs FFT 비교 | Qwen2.5-1.5B-Instruct | 둘 다 가능한 크기 |

In [ ]:
# RTX 4060 최적 설정 가이드
print("📊 RTX 4060 (8GB) 최적 학습 설정 가이드")
print("=" * 60)

configs = {
    "Qwen2.5-1.5B + FFT": {
        "method": "Full Fine-tuning",
        "precision": "fp16",
        "batch_size": 1,
        "grad_accum": 8,
        "max_seq_len": 1024,
        "vram_est": "~7GB",
        "note": "VRAM 빠듯 - 시퀀스 길이 주의"
    },
    "Qwen2.5-1.5B + LoRA": {
        "method": "LoRA (r=16, alpha=32)",
        "precision": "fp16",
        "batch_size": 2,
        "grad_accum": 4,
        "max_seq_len": 2048,
        "vram_est": "~5GB",
        "note": "여유 있음 - 권장 설정"
    },
    "Qwen2.5-3B + LoRA": {
        "method": "LoRA (r=16, alpha=32)",
        "precision": "fp16",
        "batch_size": 1,
        "grad_accum": 8,
        "max_seq_len": 1024,
        "vram_est": "~7GB",
        "note": "VRAM 빠듯"
    },
    "Qwen2.5-7B + QLoRA": {
        "method": "QLoRA (4bit NF4)",
        "precision": "4bit + fp16 compute",
        "batch_size": 1,
        "grad_accum": 8,
        "max_seq_len": 1024,
        "vram_est": "~7GB",
        "note": "VRAM 빠듯 - Unsloth 사용 권장"
    }
}

for name, config in configs.items():
    print(f"\n{'='*60}")
    print(f"🏷️ {name}")
    print(f"{'='*60}")
    for key, value in config.items():
        key_ko = {
            "method": "학습 방법",
            "precision": "정밀도",
            "batch_size": "배치 크기",
            "grad_accum": "Gradient Accumulation",
            "max_seq_len": "최대 시퀀스 길이",
            "vram_est": "예상 VRAM",
            "note": "비고"
        }.get(key, key)
        print(f"  {key_ko:>25s}: {value}")

print(f"\n\n💡 권장 사항:")
print(f"  - 실습 기본 모델: Qwen2.5-1.5B-Instruct")
print(f"  - QLoRA 실습 시: Qwen2.5-7B-Instruct + Unsloth")
print(f"  - 모든 실습에서 fp16=True 사용 (메모리 절약)")

---
## 📝 9. 정리 및 핵심 요약

### 🎯 이번 세션에서 배운 핵심 개념

| # | 개념 | 핵심 내용 |
|---|------|----------|
| 1️⃣ | **파인튜닝** | 사전학습 모델을 특정 목적에 맞게 추가 학습 |
| 2️⃣ | **학습 3단계** | Pretraining → SFT → RLHF |
| 3️⃣ | **CPT** | 도메인 텍스트로 지식 보강 (비구조화 데이터) |
| 4️⃣ | **Instruction Tuning** | (질문,답변) 쌍으로 지시 수행 능력 부여 |
| 5️⃣ | **LoRA** | 저차원 행렬 분해로 효율적 학습 (파라미터 ~1%) |
| 6️⃣ | **QLoRA** | 4bit 양자화 + LoRA (7B도 8GB에서 학습) |
| 7️⃣ | **RTX 4060 범위** | FFT: 1.5B, LoRA: 3B, QLoRA: 7B |

### 🔑 기억해야 할 핵심 수식

```
LoRA: W' = W + (alpha/r) × BA

파라미터 수: r × (d + k)  <<  d × k (원본)

QLoRA VRAM ≈ params × 0.5 bytes + LoRA overhead
```

### 📚 다음 세션 예고
- **Session 15**: 데이터 수집/증강/정제 파이프라인 - Alpaca/ShareGPT 형식, 데이터 품질 관리

In [ ]:
print("\n✅ Session 14 완료!")
print("📚 다음 세션: 데이터 수집/증강/정제 파이프라인")
print("\n🔑 퀴즈:")
print("  Q1. 7B 모델을 RTX 4060에서 학습하려면 어떤 방법을 사용해야 할까요?")
print("  Q2. LoRA의 rank가 16이고 hidden dimension이 4096일 때, 원본 대비 몇 %의 파라미터를 학습할까요?")
print("  Q3. CPT와 SFT의 가장 큰 차이점은 무엇일까요?")
print("\n💡 정답은 다음 세션 시작 시 확인합니다!")